# Search API

Search compound details by compound name, SMILES, InChI, or InChIKey.

In [1]:
import os
from pprint import pprint

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("NMRXIV_BASE_URL", "https://nmrxiv.org").rstrip("/")
API_BASE = f"{BASE_URL}/api"

session = requests.Session()
session.headers.update({"Accept": "application/json", "Content-Type": "application/json"})

BASE_URL, API_BASE

('https://nmrxiv.org', 'https://nmrxiv.org/api')

In [2]:
def api_request(method, path, **kwargs):
    url = f"{API_BASE}{path}"
    response = session.request(method, url, timeout=30, **kwargs)
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    if not response.ok:
        pprint(payload)
        response.raise_for_status()
    return payload


def as_dataframe(payload):
    if isinstance(payload, list):
        return pd.json_normalize(payload)
    if isinstance(payload, dict):
        for key in ["data", "projects", "samples", "datasets", "items", "results"]:
            if isinstance(payload.get(key), list):
                return pd.json_normalize(payload[key])
        return pd.json_normalize(payload)
    return pd.DataFrame({"value": [payload]})


def search_compound(query, query_type):
    return api_request("POST", "/v1/search", json={"query": query, "type": query_type})

## Search by InChIKey

In [3]:
result = search_compound("AAAAWQOPBUPWEV-UHFFFAOYSA-N", "InChiKey")
result

POST https://nmrxiv.org/api/v1/search -> 400
{'errors': {'type': ['The selected type is invalid.']},
 'message': 'Invalid input parameters'}


HTTPError: 400 Client Error: Bad Request for url: https://nmrxiv.org/api/v1/search

In [4]:
as_dataframe(result).head()

NameError: name 'result' is not defined

## Try text or structure searches

Change `query_type` to values such as `text`, `SMILES`, `InChi`, or `InChiKey`.

In [ ]:
query = "citronellol"
query_type = "text"

text_results = search_compound(query, query_type)
as_dataframe(text_results).head(20)